# Exploratory Data Analysis (EDA)
## AI Impact on Jobs & Layoff Risk Analysis

**Author:** Sharma Saurabh (250863131007)
**Course:** BE05000231 - Python for Data Science (GTU Semester-5)
**Month:** 2 - Analysis Phase

In [ ]:
import sys
import os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('__file__'))))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import config
from src.data_loading import load_dataset
from src.data_cleaning import handle_missing_values, remove_duplicates, standardize_column_names

print('All imports successful!')

## 1. Load Dataset

In [ ]:
df = load_dataset(config.RAW_DATASET_PATH)
print(f'Shape: {df.shape}')
df.head()

## 2. Dataset Overview

In [ ]:
print('=== Shape ===')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

print('\n=== Data Types ===')
print(df.dtypes)

print('\n=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Duplicate Rows ===')
print(f'{df.duplicated().sum()} duplicates')

## 3. Statistical Summary

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

## 4. Target Variable Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

target_counts = df['Layoff_Risk'].value_counts()
colors = ['#2ecc71', '#f39c12', '#e74c3c']

axes[0].bar(target_counts.index, target_counts.values, color=colors)
axes[0].set_title('Layoff Risk Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Risk Level')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts.values):
    axes[0].text(i, v + 100, str(v), ha='center', fontweight='bold')

axes[1].pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[1].set_title('Layoff Risk Percentage', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\nTarget Distribution:')
print(target_counts)

## 5. Numerical Features Distribution

In [ ]:
numerical_cols = ['Age', 'Years_of_Experience', 'Routine_Task_Percentage',
                  'Creativity_Requirement', 'Human_Interaction_Level',
                  'Number_of_AI_Tools_Used', 'AI_Usage_Hours_Per_Week',
                  'Tasks_Automated_Percentage', 'AI_Training_Hours']

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    axes[i].hist(df[col], bins=30, color='#3498db', edgecolor='black', alpha=0.7)
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.1f}')
    axes[i].legend()

plt.suptitle('Numerical Column Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 6. Categorical Features Distribution

In [ ]:
categorical_cols = ['Education_Level', 'Industry', 'Company_Size', 'Job_Level', 'AI_Adoption_Level']

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
colors = sns.color_palette('Set2', len(categorical_cols))

for i, col in enumerate(categorical_cols):
    value_counts = df[col].value_counts()
    axes[i].barh(value_counts.index, value_counts.values, color=colors[i])
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Count')

axes[5].set_visible(False)
plt.suptitle('Categorical Column Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Correlation Heatmap

In [ ]:
numerical_df = df.select_dtypes(include=['number'])
corr_matrix = numerical_df.corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Risk by Category

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, col in zip(axes.flatten(), ['Industry', 'Education_Level', 'Company_Size', 'AI_Adoption_Level']):
    ct = pd.crosstab(df[col], df['Layoff_Risk'], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, ax=ax, color=['#2ecc71', '#f39c12', '#e74c3c'])
    ax.set_title(f'Layoff Risk by {col.replace("_", " ").title()}', fontsize=12, fontweight='bold')
    ax.set_xlabel(col.replace('_', ' ').title())
    ax.set_ylabel('Percentage')
    ax.legend(title='Risk Level')
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.suptitle('Layoff Risk by Categories', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 9. Boxplots by Risk Level

In [ ]:
numerical_cols = ['Age', 'Years_of_Experience', 'Routine_Task_Percentage', 'Tasks_Automated_Percentage']
colors = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.flatten(), numerical_cols):
    sns.boxplot(data=df, x='Layoff_Risk', y=col, ax=ax,
               hue='Layoff_Risk', palette=colors, order=['Low', 'Medium', 'High'], legend=False)
    ax.set_title(f'{col.replace("_", " ").title()} by Risk Level', fontsize=12, fontweight='bold')

plt.suptitle('Boxplots: Numerical Features by Risk Level', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 10. AI Adoption Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ai_counts = df['AI_Adoption_Level'].value_counts()
colors = ['#3498db', '#f39c12', '#e74c3c']
axes[0].bar(ai_counts.index, ai_counts.values, color=colors)
axes[0].set_title('AI Adoption Level Distribution', fontsize=14, fontweight='bold')

scatter = axes[1].scatter(df['AI_Usage_Hours_Per_Week'],
                         df['Tasks_Automated_Percentage'],
                         c=df['Layoff_Risk'].map({'Low': 0, 'Medium': 1, 'High': 2}),
                         cmap='RdYlGn_r', alpha=0.5)
axes[1].set_title('AI Usage vs Tasks Automated', fontsize=14, fontweight='bold')
axes[1].set_xlabel('AI Usage Hours/Week')
axes[1].set_ylabel('Tasks Automated %')

plt.tight_layout()
plt.show()

## EDA Summary

- **Dataset:** 20,000 rows x 16 columns
- **Target:** Layoff_Risk (Low/Medium/High)
- **Key Findings:**
  - No missing values
  - Balanced target distribution
  - Multiple categorical and numerical features available for modeling